# NovelReader v4.2 — chỉ sửa Google Drive cho Kaggle/Colab
Giữ nguyên mã dịch, audio, cấu hình GPU 0 và batch của bản v4.2 đính kèm. Không thêm chia tải hai GPU.
Google Drive: audio lưu từng file MP3/WAV/M4A… trong `NovelReader_Exports/<truyện>/audio/<lượt xuất>/`, không tạo ZIP. TXT/EPUB giữ nguyên. Tải về máy và sao lưu vẫn có thể dùng ZIP.

- Kaggle: bật Internet và GPU. Xuất Google Drive dùng API, không gọi drive.mount của Colab. Bấm Kết nối Drive trong Web, chọn JSON OAuth **Desktop app** của bạn (bật Drive API, cấu hình consent/Test user nếu cần trong Google Cloud). Không cần sửa code.
- Mở link Google, cấp quyền rồi sao chép toàn bộ URL trả về `http://127.0.0.1:8765/?...` vào ô trong Web. Trang localhost có thể báo không kết nối được; đây là bước chuyển URL thủ công về remote app. Không gửi mã/URL/JSON vào chat.
- Colab: vẫn xin quyền mount trong tab notebook khi bấm xuất. Có thể chọn Tải về máy nếu không muốn cấu hình Drive API.
- Chạy bước 1–5; bước 1 tự tải ZIP GitHub rồi áp dụng bản vá Drive, không cần tự chọn ZIP. Giữ notebook/link/mã truy cập riêng tư.
- Quyền API lưu ở `/tmp/novelreader-drive`, ngoài Output, không kèm ZIP sao lưu. Runtime bị xóa thì cần kết nối lại. Không publish notebook/output đang chứa link truy cập còn hiệu lực.
- Dữ liệu runtime là tạm; sao lưu bước 6 trước khi kết thúc. Chưa xác minh Google OAuth/Kaggle/GPU thật trong môi trường phát triển.


## 1. Tự tải ZIP từ GitHub và cập nhật mã nguồn


In [ ]:
import os, sys, subprocess, zipfile, stat, urllib.request, hashlib, json, base64, zlib
from pathlib import Path

IS_KAGGLE = bool(os.getenv('KAGGLE_KERNEL_RUN_TYPE') or os.getenv('KAGGLE_URL_BASE') or Path('/kaggle/working').is_dir())
BASE_DIR = Path('/kaggle/working' if IS_KAGGLE else '/content')
BASE_DIR.mkdir(parents=True, exist_ok=True)
REPO_DIR = BASE_DIR / 'NovelReader_T4_Web'
SOURCE_URL = 'https://raw.githubusercontent.com/dungbeommm/NovelReader_Translate/main/NovelReader_GPU_notebook.zip'
for name in ('server_process', 'tunnel_process'):
    proc = globals().get(name)
    if proc is not None and proc.poll() is None:
        raise RuntimeError('Dừng cell Web/tunnel trước khi tải/cập nhật mã nguồn.')
# Keep checkpoints from earlier Colab-style notebooks run on Kaggle.
legacy = Path('/content/NovelReader_T4_Web')
if IS_KAGGLE and (legacy / 'data/novel.sqlite3').is_file() and not (REPO_DIR / 'data/novel.sqlite3').is_file():
    REPO_DIR = legacy
    print('Giữ thư mục/checkpoint cũ:', REPO_DIR, '— nhớ sao lưu bước 6 trước khi kết thúc Kaggle.')
archive = BASE_DIR / 'NovelReader_GPU_notebook.zip'
partial = archive.with_suffix('.zip.part')
print('Đang tải:', SOURCE_URL)
try:
    with urllib.request.urlopen(SOURCE_URL, timeout=90) as src, partial.open('wb') as out:
        total = 0
        while True:
            chunk = src.read(1024 * 1024)
            if not chunk:
                break
            total += len(chunk)
            if total > 128 * 1024 * 1024:
                raise RuntimeError('ZIP mã nguồn vượt giới hạn 128 MB.')
            out.write(chunk)
    if not zipfile.is_zipfile(partial):
        raise RuntimeError('Đường dẫn GitHub không trả về ZIP hợp lệ. Kiểm tra file còn public.')
    os.replace(partial, archive)
finally:
    partial.unlink(missing_ok=True)
print('ZIP SHA256:', hashlib.sha256(archive.read_bytes()).hexdigest())
with zipfile.ZipFile(archive) as z:
    infos = z.infolist()
    if len(infos) > 5000 or sum(i.file_size for i in infos) > 512 * 1024 * 1024:
        raise RuntimeError('ZIP có quá nhiều tệp hoặc dung lượng giải nén quá lớn.')
    markers = [n for n in z.namelist() if n == 'main.py' or (n.endswith('/main.py') and len(Path(n).parts) == 2)]
    if len(markers) != 1:
        raise RuntimeError('Không xác định được thư mục dự án trong ZIP.')
    prefix = markers[0][:-len('main.py')]
    for info in infos:
        if not info.filename.startswith(prefix):
            continue
        rel = info.filename[len(prefix):]
        if not rel:
            continue
        target = (REPO_DIR / rel).resolve()
        if not target.is_relative_to(REPO_DIR.resolve()) or stat.S_ISLNK(info.external_attr >> 16):
            raise RuntimeError('ZIP có đường dẫn hoặc symlink không an toàn.')
        # Updating code must never replace user's data, weights, or credentials.
        if Path(rel).parts[0] in {'data', 'models', 'output_text', 'output_audio', '.git'} or Path(rel).name == '.env':
            continue
        if info.is_dir():
            target.mkdir(parents=True, exist_ok=True)
        else:
            target.parent.mkdir(parents=True, exist_ok=True)
            with z.open(info) as src, target.open('wb') as dst:
                import shutil
                shutil.copyfileobj(src, dst)
# Source-only overlay maintains v4 functionality even if GitHub currently serves v3.
OVERLAY_V4 = '...'
for rel, content in json.loads(zlib.decompress(base64.b64decode(OVERLAY_V4))).items():
    path = REPO_DIR / rel
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(content, encoding='utf-8')
os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))
print('Đã nạp bản Web có xuất Drive. Thư mục:', REPO_DIR)
subprocess.run(['nvidia-smi'], check=True)

## 2. Cài thư viện (không thay PyTorch CUDA có sẵn của Colab)
Chỉ dùng CT2 nên không cần tải lại PyTorch. Bước này cài requirements trước, sau đó gỡ hai bản ORT và chỉ cài lại ORT GPU.
Nếu trước đó đã import ONNX Runtime/Piper/CT2 trong kernel, hãy **Restart session**, rồi chạy lại từ bước 1. Không đổi runtime sang CPU.

In [ ]:
if any(module in sys.modules for module in ('onnxruntime', 'piper', 'ctranslate2')):
    raise RuntimeError('Đã import GPU runtime. Restart session trước khi cài lại để tránh giữ thư viện cũ trong bộ nhớ.')
subprocess.run([sys.executable, 'scripts/setup_colab_runtime.py'], check=True)
subprocess.run(['apt-get', 'update', '-qq'], check=True)
subprocess.run(['apt-get', 'install', '-y', '-qq', 'ffmpeg', 'espeak-ng'], check=True)

## 3. Cấu hình T4
Web mặc định batch **96** (cho phép chỉnh đến 256), CT2 nội bộ **96**, beam **2**. Thử beam 1 trong Web nếu chấp nhận đánh đổi chất lượng.
Pipeline audio có công tắc trong Web để so sánh với chế độ tuần tự. Nếu VRAM thiếu, giảm batch 96 → 64 → 32.
Piper CUDA là bắt buộc ở notebook này, không chuyển CPU âm thầm. Để chủ động thử CPU TTS: đặt PIPER_DEVICE=cpu và PIPER_REQUIRE_CUDA=0, khởi động lại server; smoke check bước 4 khi đó sẽ không phù hợp vì nó kiểm tra CUDA bắt buộc.

In [ ]:
import site
os.environ.update({
    'HACHIMIMT_WEB_BATCH_SIZE': '96',
    'HACHIMIMT_BATCH_SIZE': '96',
    'HACHIMIMT_CT2_BATCH_TYPE': 'tokens',
    'HACHIMIMT_CT2_WINDOW_MULTIPLIER': '16',
    'HACHIMIMT_INTER_THREADS': '1',
    'HACHIMIMT_COMPUTE_TYPE': 'int8_float16',
    'HACHIMIMT_FORCE_CT2_CUDA': '1',
    'HACHIMIMT_GPU_INDICES': '0',
    'HACHIMIMT_AUTO_ALL_GPUS': '0',
    'HACHIMIMT_TOKENIZE_WORKERS': str(min(os.cpu_count() or 2, 4)),
    'PIPER_DEVICE': 'cuda', 'PIPER_REQUIRE_CUDA': '1',
    'ALLOWED_HOSTS': '*', 'ALLOW_TUNNEL': '1', 'COLAB_GPU': '1',
    'NOVEL_DATA': str(REPO_DIR / 'data'),
    'NOVEL_MODELS': str(REPO_DIR / 'models/translation'),
    'TTS_VOICE_DIR': str(REPO_DIR / 'models/voices'),
    'NOVEL_OUTPUT_TEXT': str(REPO_DIR / 'output_text'),
    'NOVEL_OUTPUT_AUDIO': str(REPO_DIR / 'output_audio'),
    'PYTHONUNBUFFERED': '1',
    'NOVEL_COLAB_DRIVE_BRIDGE': '1',
    'NOVEL_COLAB_DRIVE_ROOT': '/content/drive/MyDrive',
})
# Export NVIDIA wheel libraries before starting child processes.
libdirs = []
for base in site.getsitepackages():
    libdirs.extend(str(p) for p in (Path(base) / 'nvidia').glob('*/lib') if p.is_dir())
os.environ['LD_LIBRARY_PATH'] = ':'.join(libdirs + [os.environ.get('LD_LIBRARY_PATH', '')])
for name in ('data/logs', 'models/translation', 'models/voices', 'output_text', 'output_audio'):
    (REPO_DIR / name).mkdir(parents=True, exist_ok=True)
print('Đã đặt cấu hình. Chạy bước 4 rồi bước 5.')
# Drive fix only: preserve v4.2 GPU/batch settings above.
os.environ['NOVEL_DRIVE_BACKEND'] = 'api' if IS_KAGGLE else 'colab'
os.environ['NOVEL_COLAB_DRIVE_BRIDGE'] = '0' if IS_KAGGLE else '1'
os.environ['NOVEL_DRIVE_CREDENTIAL_DIR'] = '/tmp/novelreader-drive'
print('Drive:', 'API OAuth trong Web (cần Desktop OAuth JSON)' if IS_KAGGLE else 'mount trong tab Colab')


## 4. Tải model cần dùng và kiểm tra GPU thật
Chỉ tải giọng mặc định, không tải toàn bộ kho giọng. Script dịch một câu và tạo WAV ngắn; nó dừng nếu CT2/Piper không dùng CUDA.
Nếu bước này lỗi, xem thông báo CUDA/cuDNN; đừng tiếp tục chạy Web rồi tưởng là đã dùng GPU.

In [ ]:
subprocess.run([sys.executable, 'scripts/check_gpu.py',
                '--model', 'HachimiMT-60-QT', '--voice', 'ngochuyennew'], check=True)

## 5. Web + Drive đúng môi trường
Giữ cell này chạy để phục vụ Web. Kaggle dùng Kết nối Drive trong Web, không mount Colab. Nếu chưa có OAuth client, có thể tải kết quả về máy. Colab vẫn yêu cầu quyền mount trong tab notebook khi bấm xuất. Không dừng runtime trong lúc xuất file lớn.


In [ ]:
import time, re, urllib.request, signal, secrets
if not IS_KAGGLE:
    from google.colab import drive
from gdrive.colab_bridge import process_drive_requests
from IPython.display import display, Markdown

for name in ('server_process', 'tunnel_process'):
    old = globals().get(name)
    if old is not None and old.poll() is None:
        raise RuntimeError('Server/tunnel cũ còn chạy. Dừng cell cũ trước.')
# Fail early if port 8000 is already occupied.
import socket
with socket.socket() as sock:
    sock.bind(('127.0.0.1', 8000))
cloudflared = BASE_DIR / 'cloudflared'
if not cloudflared.is_file():
    urllib.request.urlretrieve('https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64', cloudflared)
    cloudflared.chmod(0o755)
# Per-session secret is never written to the ZIP or Drive.
os.environ['NOVEL_WEB_ACCESS_TOKEN'] = secrets.token_urlsafe(32)
os.environ['NOVEL_EXPORT_SESSION_ID'] = secrets.token_hex(16)
server_log = REPO_DIR / 'data/logs/colab-server.log'
tunnel_log = REPO_DIR / 'data/logs/colab-tunnel.log'
server_process = tunnel_process = None
try:
    with server_log.open('w') as sf, tunnel_log.open('w') as tf:
        server_process = subprocess.Popen(
            [sys.executable, '-m', 'uvicorn', 'main:app', '--host', '127.0.0.1',
             '--port', '8000', '--workers', '1', '--no-access-log', '--timeout-graceful-shutdown', '120'],
            cwd=REPO_DIR, stdout=sf, stderr=subprocess.STDOUT)
        for _ in range(120):
            if server_process.poll() is not None:
                raise RuntimeError(server_log.read_text()[-6000:])
            try:
                with urllib.request.urlopen(urllib.request.Request('http://127.0.0.1:8000/api/state', headers={'X-Novel-Access': os.environ['NOVEL_WEB_ACCESS_TOKEN']}), timeout=2):
                    break
            except Exception:
                time.sleep(1)
        else:
            raise RuntimeError('Web chưa sẵn sàng. Xem colab-server.log.')
        tunnel_process = subprocess.Popen(
            [str(cloudflared), 'tunnel', '--url', 'http://127.0.0.1:8000'],
            stdout=tf, stderr=subprocess.STDOUT)
        public_url = None
        for _ in range(90):
            if tunnel_process.poll() is not None:
                raise RuntimeError(tunnel_log.read_text()[-4000:])
            match = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', tunnel_log.read_text())
            if match:
                public_url = match.group(0)
                break
            time.sleep(1)
        if not public_url:
            raise RuntimeError('Không lấy được URL tunnel. Xem colab-tunnel.log.')
        display(Markdown(f"### [Mở NovelReader]({public_url}/?access_token={os.environ['NOVEL_WEB_ACCESS_TOKEN']})\nLink riêng của phiên — không chia sẻ. Kaggle: bấm Kết nối Drive trong Web. Colab: quay lại notebook để cấp quyền mount nếu được hỏi."))
        print("Mã truy cập (dán vào trang đăng nhập nếu cần):", os.environ["NOVEL_WEB_ACCESS_TOKEN"])
        print("Địa chỉ Web:", public_url)
        with server_log.open() as log:
            while server_process.poll() is None and tunnel_process.poll() is None:
                if not IS_KAGGLE:
                    process_drive_requests(REPO_DIR / 'data/drive_exports', drive.mount)
                line = log.readline()
                if line:
                    print(line, end='')
                else:
                    time.sleep(.2)
except KeyboardInterrupt:
    print('Đang dừng. Không đóng runtime cho tới khi hoàn tất...')
finally:
    for proc in (tunnel_process, server_process):
        if proc is not None and proc.poll() is None:
            proc.terminate()
    for proc in (tunnel_process, server_process):
        if proc is not None and proc.poll() is None:
            try:
                proc.wait(timeout=135)
            except subprocess.TimeoutExpired:
                proc.kill()
                proc.wait()
                print('Đã buộc dừng quá thời hạn; audio chưa xong sẽ được làm lại khi tiếp tục.')
    print('Đã dừng Web và tunnel.')

## 6. Tải kết quả sau khi dừng Web
Gói gồm text/audio và database checkpoint. Không đưa token/cấu hình Google Drive vào bản sao lưu này. Sao lưu chỉ nhất quán khi server đã dừng.

In [ ]:
if globals().get('server_process') is not None and server_process.poll() is None:
    raise RuntimeError('Dừng cell Web trước khi đóng gói checkpoint.')
from datetime import datetime
backup = BASE_DIR / ('NovelReader_outputs_' + datetime.now().strftime('%Y%m%d_%H%M%S') + '.zip')
with zipfile.ZipFile(backup, 'w', zipfile.ZIP_DEFLATED) as z:
    for folder in ('output_text', 'output_audio'):
        for p in (REPO_DIR / folder).rglob('*'):
            if p.is_file() and not any(part.startswith('.') for part in p.relative_to(REPO_DIR).parts):
                z.write(p, p.relative_to(REPO_DIR))
    # SQLite backup API also includes committed WAL data.
    import sqlite3, tempfile
    db = REPO_DIR / 'data/novel.sqlite3'
    if db.exists():
        with tempfile.TemporaryDirectory() as temp:
            snapshot = Path(temp) / 'novel.sqlite3'
            with sqlite3.connect(db) as src, sqlite3.connect(snapshot) as dst:
                src.backup(dst)
            z.write(snapshot, 'data/novel.sqlite3')
print('Bản sao lưu:', backup.name)
if IS_KAGGLE:
    from IPython.display import display, FileLink
    display(FileLink(str(backup)))
    print('Hoặc tải ZIP từ mục Output/Files của Kaggle. Không xuất /tmp hoặc file quyền OAuth.')
else:
    from google.colab import files
    files.download(str(backup))

## Đo tốc độ (tùy chọn, dừng Web trước)
Dùng `translation/core/benchmark_sweep.py` với cùng file TXT để so batch/beam/compute type. Đây là benchmark core dịch, không đại diện thời gian Web + TTS.
Trong log Web có `TIMING translation` và `TIMING audio` theo chương. So sánh cùng vài chương, cùng giọng, cùng thiết lập để tránh kết luận sai.